In [18]:
# trying to run code locally from: https://colab.research.google.com/github/google-research/weatherbenchX/blob/main/docs/source/wbx_quickstart.ipynb#scrollTo=0c7ee5de-bac3-43e5-adda-e82f3b4160c6
import apache_beam as beam
import numpy as np
import xarray as xr
import weatherbenchX
from weatherbenchX.data_loaders import xarray_loaders
from weatherbenchX.metrics import deterministic
from weatherbenchX.metrics import base as metrics_base
from weatherbenchX import aggregation
from weatherbenchX import weighting
from weatherbenchX import binning
from weatherbenchX import time_chunks
from weatherbenchX import beam_pipeline

from datetime import datetime, timedelta

In [14]:
prediction_path = 'gs://weatherbench2/datasets/pangu/2018-2022_0012_0p25.zarr'
target_path = 'gs://weatherbench2/datasets/era5/1959-2023_01_10-full_37-1h-0p25deg-chunk-1.zarr'

In [15]:
full_surface_var_list = ["2m_temperature", "10m_u_component_of_wind", "10m_v_component_of_wind"] 
full_atm_var_list = ["geopotential", "v_component_of_wind", "u_component_of_wind", "specific_humidity", "temperature"]

variables = full_surface_var_list + full_atm_var_list
target_data_loader = xarray_loaders.TargetsFromXarray(
    path=target_path,
    variables=variables,
)
prediction_data_loader = xarray_loaders.PredictionsFromXarray(
    path=prediction_path,
    variables=variables,
)

/Users/ohouck/vc/ai_weather_ag/.venv/lib/python3.11/site-packages/weatherbenchX/data_loaders/xarray_loaders.py:108: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  self._ds = xr.open_zarr(path)


In [38]:
start_date = datetime.strptime("2018-01-01", '%Y-%m-%d')
end_date = datetime.strptime("2018-02-01", '%Y-%m-%d')

date_list= np.arange(
    np.datetime64(start_date), 
    np.datetime64(end_date), 
    dtype='datetime64[D]'
)
init_times = np.array(date_list, dtype='datetime64[ns]')
# lead_times = np.array([24], dtype='timedelta64[h]').astype('timedelta64[ns]')   # To silence xr warnings.
lead_times = np.array([24, 48, 72, 96, 120, 144, 168], dtype='timedelta64[h]').astype('timedelta64[ns]')   # To silence xr warnings.
full_lat_values = np.arange(16.75, 17.25, 0.25)
full_lon_values = np.arange(71.75, 72.25, 0.25)

In [39]:
target_chunk = target_data_loader.load_chunk(init_times, lead_times)
target_chunk

# 1 day 1 lead time: 27.9 sec
# 1 day 7 lead times: 3m 1.6 sec
# 3 day 1 lead time: 1m 26.1 sec
# 3 days 7 lead_times 3min 53.3 sec

<xarray.Dataset> Size: 16GB
Dimensions:                  (latitude: 721, longitude: 1440, init_time: 3,
                              lead_time: 7, level: 37)
Coordinates:
  * latitude                 (latitude) float32 3kB 90.0 89.75 ... -89.75 -90.0
  * longitude                (longitude) float32 6kB 0.0 0.25 ... 359.5 359.8
    valid_time               (init_time, lead_time) datetime64[ns] 168B 2018-...
  * init_time                (init_time) datetime64[ns] 24B 2018-01-01 ... 20...
  * lead_time                (lead_time) timedelta64[ns] 56B 1 days ... 7 days
  * level                    (level) int64 296B 1 2 3 5 7 ... 925 950 975 1000
Data variables:
    2m_temperature           (init_time, lead_time, latitude, longitude) float32 87MB ...
    10m_u_component_of_wind  (init_time, lead_time, latitude, longitude) float32 87MB ...
    10m_v_component_of_wind  (init_time, lead_time, latitude, longitude) float32 87MB ...
    geopotential             (init_time, lead_time, level, latitude, longitude) float32 3GB ...
    v_component_of_wind      (init_time, lead_time, level, latitude, longitude) float32 3GB ...
    u_component_of_wind      (init_time, lead_time, level, latitude, longitude) float32 3GB ...
    specific_humidity        (init_time, lead_time, level, latitude, longitude) float32 3GB ...
    temperature              (init_time, lead_time, level, latitude, longitude) float32 3GB ...
Attributes:
    long_name:   2 metre temperature
    short_name:  t2m
    units:       K

In [ ]:

prediction_chunk = prediction_data_loader.load_chunk(init_times, lead_times)
prediction_chunk

In [7]:
metrics = {
  'rmse': deterministic.RMSE(),
  'mae': deterministic.MAE(),
}
metrics['rmse'].statistics

statistics = metrics_base.compute_unique_statistics_for_all_metrics(
  metrics, prediction_chunk, target_chunk
)
statistics.keys()

dict_keys(['SquaredError', 'AbsoluteError'])